# 01 · Field Validation — Satellite NDSI vs Ground EC

We pair the 162 in-situ electrical-conductivity (EC) measurements from the March 2024 field campaign with Landsat 8 NDSI extracted at each sampling coordinate from Google Earth Engine. Two regressions are run — linear (raw units) and log–log — and the canonical validation scatter plot is generated.

**Expected outputs** (matching the manuscript):

- Linear fit: **R² ≈ 0.249**, p < 0.001, n = 162
- Log–log fit: **R² ≈ 0.273** (modest improvement consistent with NDSI's compressive response at higher salinity)
- Figure: `figures/fig_validation.png`

In [ ]:
import sys
from pathlib import Path

# Make the local `src` package importable
sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings('ignore')

import ee
import pandas as pd

from src.config import DATA_DIR, FIGURES_DIR, DEFAULT_FIELD_DATA_FILE
from src.gee_ndsi import initialize_ee, build_ndsi_collection, sample_points
from src.validation import (
    load_field_ec,
    compare_linear_vs_log,
    plot_validation_scatter,
)

## 1 · Initialize Earth Engine

Run `earthengine authenticate` from the terminal once before running this cell. Set `GEE_PROJECT_ID` in `src/config.py` or pass `project_id=` here.

In [ ]:
initialize_ee()  # or: initialize_ee(project_id='your-gee-project-id')
print('GEE connected ✓')

## 2 · Load the field EC dataset

The MIT March-2024 dataset uses a two-row header — `load_field_ec` handles that. Expect 162 rows after dropping any missing coordinates / EC.

In [ ]:
ec_path = DATA_DIR / DEFAULT_FIELD_DATA_FILE
df_ec = load_field_ec(ec_path)

print(f'Total EC samples: {len(df_ec)}')
print(f'EC range:         {df_ec["EC"].min():.4f} - {df_ec["EC"].max():.4f} mS/cm')
print(f'Lat range:        {df_ec["Latitude"].min():.3f} - {df_ec["Latitude"].max():.3f}')
print(f'Lng range:        {df_ec["Longitude"].min():.3f} - {df_ec["Longitude"].max():.3f}')
df_ec.head()

## 3 · Extract co-located NDSI from Landsat 8 (Feb – Apr 2024 window)

The two-month envelope around field-sampling provides enough cloud-free scenes for a stable per-point mean NDSI while staying close to the sampling date.

In [ ]:
from datetime import date

coll = build_ndsi_collection(
    start=date(2024, 2, 1),
    end=date(2024, 4, 30),
    platforms=('L8',),
)

# Reduce to a single mean image per point. We use the explicit
# per-point sampler from src.gee_ndsi for parity with the field data.
ts = sample_points(coll, df_ec, scale=30)

# One NDSI value per (point) by averaging across the window's scenes
per_point = (
    ts.groupby(['ID', 'Latitude', 'Longitude'])['ndsi']
      .mean()
      .reset_index()
      .rename(columns={'ndsi': 'NDSI'})
)

df_valid = df_ec.merge(per_point[['ID', 'NDSI']], on='ID', how='inner')
df_valid = df_valid.dropna(subset=['NDSI', 'EC'])
print(f'Valid pairs:      {len(df_valid)}')
print(f'NDSI range:       {df_valid["NDSI"].min():.4f} - {df_valid["NDSI"].max():.4f}')

## 4 · Linear vs log–log regression

In [ ]:
stats = compare_linear_vs_log(df_valid, x_col='NDSI', y_col='EC')
print(stats.to_string(index=False))

improvement = stats.iloc[1]['r2'] - stats.iloc[0]['r2']
print(f'\nLog transformation R² improvement: +{improvement:.3f}')

## 5 · Validation scatter (Figure 1)

In [ ]:
fig_path = FIGURES_DIR / 'fig_validation.png'
fig, ax, s = plot_validation_scatter(
    df_valid,
    x_col='NDSI', y_col='EC',
    save_to=fig_path,
)
print(f'Saved: {fig_path}')
print(f'R² = {s["r2"]:.3f}  ·  R = {s["r"]:.3f}  ·  p = {s["p_value"]:.6f}  ·  n = {s["n"]}')

---

**Validation complete.** The R² of ~0.249 is modest but statistically very significant (p < 0.001). The log-transformed model improves the fit slightly — consistent with NDSI's documented compressive response at higher salt concentrations.

The validation supports using NDSI as a *relative* salinity proxy for the long-term reconstruction in notebook **02** and the projection work in notebooks **05** and **06**. The downstream pipeline does **not** translate NDSI to absolute EC — projections are expressed in NDSI units, with the relationship documented here as supporting evidence.

**Next:** `02_gee_long_time_series.ipynb` — the 26-year multi-sensor NDSI reconstruction (2000 – 2026).